# Optimizing Tool Descriptions

This notebook accompanies Chapter 10 §10.4.3 of *Context Engineering with DSPy*: Optimizing Tool Descriptions.

## Required environment variables

Add these to a `.env` at the repo root (see `.env.example`):

- `OPENAI_API_KEY` — used as the task / judge / router LM via `dspy.LM("openai/...")`.
- `OPENROUTER_API_KEY` — used when the notebook calls Anthropic models via OpenRouter (`dspy.LM("openrouter/anthropic/claude-opus-4.7")`). No Claude Code CLI is required.

## Original source

Imported from the writing repo at `working/optimize-tool-descriptions.ipynb`.


In [ ]:
%pip install -r ../requirements.txt -q

# Optimizing tool descriptions for an agent

When an agent has six tools to choose from, the only thing standing between the user's request and the right tool is the description string somebody wrote in a hurry. Most of these are vague ("Search the web."), overlapping (`read_file` vs `cat_file`), or written for the developer who *added* the tool rather than the agent that has to *choose* it. We're going to optimize those descriptions against a small benchmark of (task -> correct tool) pairs and measure tool-selection accuracy before and after.

**How it works:**
- **Candidate** — a JSON dict of `{tool_name: description}`. GEPA rewrites the descriptions.
- **Dataset** — short tasks plus the tool that should be called for each.
- **Evaluator** — feed the descriptions + the task into a router LM, see which tool it picks, score 1 or 0.
- **Reflection LM** — reads the failures and proposes a clearer set of descriptions.

**Prerequisites:**
- `OPENAI_API_KEY` in your environment

In [ ]:
import os, json, dspy
from gepa.optimize_anything import (
    optimize_anything, GEPAConfig, EngineConfig, ReflectionConfig,
)

assert os.environ.get("OPENAI_API_KEY"), "set OPENAI_API_KEY first"

ROUTER_LM = dspy.LM("openai/gpt-5-mini", temperature=0.0)
REFLECTION_MODEL = "openai/gpt-5"

## Seed: weak descriptions and a benchmark

Five tools, descriptions written badly on purpose -- they overlap, they're terse, they describe *what the tool does* rather than *when to call it*. The benchmark is twelve tasks paired with the tool that should run.

In [ ]:
SEED_TOOLS = {
    "read_file":   "Read a file.",
    "grep":        "Search for text.",
    "git_diff":    "Run git diff.",
    "web_search":  "Search the web.",
    "run_tests":   "Run tests.",
}

BENCH = [
    ("open package.json and tell me the version",                      "read_file"),
    ("show me the contents of src/main.py",                            "read_file"),
    ("find every place we call requests.post",                         "grep"),
    ("which files import the old auth_legacy module",                  "grep"),
    ("what did I change since my last commit",                         "git_diff"),
    ("summarise the unstaged changes in this repo",                    "git_diff"),
    ("what's the latest release of the openai python sdk",             "web_search"),
    ("is there a known cve for log4j 2.14.1",                          "web_search"),
    ("did the test suite pass on this branch",                         "run_tests"),
    ("run the integration tests for the billing module",               "run_tests"),
    ("find the failing assertion in tests/test_invoice.py",            "grep"),
    ("compare the current branch against main",                        "git_diff"),
]

DATASET = [{"task": t, "correct_tool": c} for t, c in BENCH]

## The router

One tiny LM call. Hand the model the current set of descriptions plus the task and ask it to name a tool. The router never changes during optimization -- only the descriptions do, and that's what we're measuring.

In [ ]:
def route(descriptions: dict, task: str) -> str:
    listing = "\n".join(f"- {name}: {desc}" for name, desc in descriptions.items())
    prompt = (
        "You are routing a task to one tool. Tools available:\n"
        f"{listing}\n\n"
        f"Task: {task}\n\n"
        "Reply with ONLY the tool name. Nothing else."
    )
    return ROUTER_LM(messages=[{"role": "user", "content": prompt}])[0].strip()

## Baseline accuracy

How often does the router pick the right tool with the weak descriptions? Run it once and remember the number.

In [ ]:
def accuracy(descriptions: dict) -> float:
    correct = sum(
        1 for row in DATASET
        if route(descriptions, row["task"]) == row["correct_tool"]
    )
    return correct / len(DATASET)

baseline = accuracy(SEED_TOOLS)
print(f"baseline: {baseline:.0%} ({int(baseline*len(DATASET))}/{len(DATASET)})")

## Evaluator

The candidate the optimizer mutates is a JSON dict of `{tool_name: description}`. Each rollout routes one task and returns 1 or 0. The side info tells the reflection LM *what it picked* vs *what it should have picked* -- that's the per-example gradient GEPA needs.

In [ ]:
def evaluator(candidate: str, example: dict) -> tuple[float, dict]:
    descriptions = json.loads(candidate)
    picked = route(descriptions, example["task"])
    correct = picked == example["correct_tool"]
    return (
        1.0 if correct else 0.0,
        {
            "task":         example["task"],
            "picked":       picked,
            "should_pick":  example["correct_tool"],
            "current_desc": descriptions.get(example["correct_tool"], "<missing>"),
        },
    )

## Smoke-test budget

The book runs this with `max_metric_calls=60`. **This cell is capped to 3** so the notebook completes quickly; expect little or no improvement at this budget. The next cell holds the full-budget version (commented out) for when you want to reproduce the chapter's results.


In [ ]:
result = optimize_anything(
    seed_candidate=json.dumps(SEED_TOOLS, indent=2),
    evaluator=evaluator,
    dataset=DATASET,
    objective=(
        "Rewrite the tool descriptions so the router picks the right tool for each task. "
        "Each description should say WHEN to use the tool, not just WHAT it does. "
        "Output a single JSON object: {tool_name: description}. No preamble."
    ),
    config=GEPAConfig(
        engine=EngineConfig(
            run_dir="outputs/tool-descriptions",
            max_metric_calls=3,
            display_progress_bar=True,
            track_best_outputs=True,
        ),
        reflection=ReflectionConfig(reflection_lm=REFLECTION_MODEL),
    ),
)

optimized_tools = json.loads(result.best_candidate)
print(json.dumps(optimized_tools, indent=2))

In [ ]:
# Full reproduction of the book's §10.4.3 result.
# Uncomment to run with the chapter's full budget.
#
# result = optimize_anything(
#     seed_candidate=json.dumps(SEED_TOOLS, indent=2),
#     evaluator=evaluator,
#     dataset=DATASET,
#     objective=(
#         "Rewrite the tool descriptions so the router picks the right tool for each task. "
#         "Each description should say WHEN to use the tool, not just WHAT it does. "
#         "Output a single JSON object: {tool_name: description}. No preamble."
#     ),
#     config=GEPAConfig(
#         engine=EngineConfig(
#             run_dir="outputs/tool-descriptions",
#             max_metric_calls=60,
#             display_progress_bar=True,
#             track_best_outputs=True,
#         ),
#         reflection=ReflectionConfig(reflection_lm=REFLECTION_MODEL),
#     ),
# )
#
# optimized_tools = json.loads(result.best_candidate)
# print(json.dumps(optimized_tools, indent=2))

In [ ]:
after = accuracy(optimized_tools)
print(f"baseline:  {baseline:.0%}")
print(f"optimized: {after:.0%}")
print(f"delta:     {(after - baseline)*100:+.0f} pp")

Drop the optimized descriptions into the docstrings of your actual tool functions and you've upgraded every agent that loads them. Same loop works for MCP tool descriptions, CLI sub-command help strings, and the natural-language slug your skills use to advertise themselves to the router.